In [ ]:
from dotenv import load_dotenv
import os
import glob
import time

load_dotenv()

In [ ]:
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
PDF_DIR = "document"

def read_docs(directory):
    docs = []
    for path in sorted(glob.glob(f"{directory}/*.pdf")):
        reader = PdfReader(path)
        pages = []
        for page in reader.pages:
            text = page.extract_text() or ""
            if text.strip():
                pages.append(text)
        content = "\n\n".join(pages)
        if content.strip():
            docs.append(Document(page_content=content, metadata={"source": path}))
    return docs

docs = read_docs(PDF_DIR)
print("documents:", len(docs))
if not docs:
    raise ValueError(f"No PDFs found in {PDF_DIR}")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print("chunks:", len(chunks))

In [ ]:
embeddings = HuggingFaceEmbeddings(
  model_name="sentence-transformers/all-MiniLM-L6-v2"
)

sample_vector = embeddings.embed_query("test")
print("embedding_dimension:", len(sample_vector))

In [ ]:
if "embeddings" not in globals():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

if "sample_vector" not in globals():
    sample_vector = embeddings.embed_query("test")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY in your .env file")

index_name = "myproject"
pc = Pinecone(api_key=PINECONE_API_KEY)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(sample_vector),
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(1)

vector_store = PineconeVectorStore(index=pc.Index(index_name), embedding=embeddings)

for i in range(0, len(chunks), 50):
    vector_store.add_documents(chunks[i:i + 50])

print("uploaded:", len(chunks))

In [ ]:
GROQ_API_KEY = os.getenv("GROQ")
if not GROQ_API_KEY:
    raise ValueError("Set GROQ in your .env file")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
)

def ask(question):
    result = qa.invoke({"query": question})
    return result["result"]

In [ ]:
print(ask("What is the difference between an asset and a liability?"))